# BLUP Heritability and Trait Network Analysis

## Rice Mutant Population Analysis Pipeline

This notebook performs a comprehensive trait analysis pipeline used in plant breeding and quantitative genetics research.

### Analyses included

- BLUP estimation using mixed linear models (REML)
- Variance component estimation
- BLUP-based heritability
- Partial correlation network (Graphical Lasso)
- Trait interaction network
- Yield predictor identification using regression

### Outputs

Tables

- BLUP matrix
- Variance components
- Standardized BLUP matrix
- Partial correlation matrix
- Hub trait centrality
- Yield predictor importance

Figures

- Partial correlation heatmap
- Trait interaction network
- Yield predictor importance plot

All figures exported at **600 dpi journal quality**.

---

## Dataset Requirements

Dataset must contain the following columns:

- Genotype
- Replication

Trait columns such as:

- Days to flowering
- Days to maturity
- Plant height
- Tillers per hill
- Effective tillers per hill
- Panicle length
- Primary branch per panicle
- Secondary branch per panicle
- Filled grain per panicle
- Grain length
- Grain breadth
- Grain yield per hill
- Straw yield per hill
- Harvest index


## Step 1 — Install Required Libraries

This installs packages needed for mixed models, machine learning, and network analysis.

In [ ]:
!pip install statsmodels scikit-learn networkx openpyxl

## Step 2 — Import Libraries

In [ ]:
import os
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
from sklearn.linear_model import LinearRegression

import networkx as nx
from google.colab import files

warnings.filterwarnings("ignore")

## Step 3 — Output Folder

All results will be stored in this directory.

In [ ]:
OUTDIR = "BLUP_Network_YieldDrivers_outputs"
os.makedirs(OUTDIR, exist_ok=True)

THRESHOLD_PC = 0.20
RANDOM_SEED = 42

## Step 4 — Trait Abbreviations

Used for plotting and labeling figures.

In [ ]:
abbr = {
 'Days to flowering':'DF',
 'Days to maturity':'DM',
 'Plant height':'PH',
 'Tillers per hill':'TH',
 'Effective tillers per hill':'ETH',
 'Panicle length':'PL',
 'Primary branch per panicle':'PBP',
 'Secondary branch per panicle':'SBP',
 'Filled grain per panicle':'FG',
 'Grain length':'GL',
 'Grain breadth':'GB',
 '1000 grain weight':'TGW',
 'Grain yield per hill':'GYH',
 'Straw yield per hill':'SYH',
 'Harvest index':'HI'
}

def safe_abbr(trait):
    return abbr.get(trait, re.sub(r"[^A-Za-z0-9]+", "", trait)[:6])

## Step 5 — Upload Dataset

In [ ]:
print("Upload Excel dataset")
uploaded = files.upload()

infile = list(uploaded.keys())[0]

df = pd.read_excel(infile)

df.columns = [c.strip() for c in df.columns]

df['Genotype'] = df['Genotype'].astype(str)
df['Replication'] = df['Replication'].astype(str)

traits = [c for c in df.columns if c not in ['Genotype','Replication']]

print("Traits detected:", len(traits))

## Step 6 — BLUP Estimation

In [ ]:
genotypes = sorted(df['Genotype'].unique())
reps = sorted(df['Replication'].unique())
r = len(reps)

blup = pd.DataFrame(index=genotypes, columns=traits)
vc_rows = []

for t in traits:
    formula = f'Q("{t}") ~ C(Replication)'
    md = smf.mixedlm(formula, df, groups=df['Genotype'])

    try:
        m = md.fit(reml=True, method='lbfgs')
    except:
        m = md.fit(reml=True, method='powell')

    mu = float(m.fe_params.iloc[0])
    reffs = m.random_effects

    for g in genotypes:
        u = float(np.asarray(reffs.get(g,[0]))[0])
        blup.loc[g,t] = mu + u

    sigma_g2 = float(m.cov_re.iloc[0,0])
    sigma_e2 = float(m.scale)

    H2 = sigma_g2/(sigma_g2 + sigma_e2/r)

    vc_rows.append({
        'Trait':t,
        'Trait_abbr':safe_abbr(t),
        'sigma_g2':sigma_g2,
        'sigma_e2':sigma_e2,
        'H2_BLUP':H2
    })

vc_df = pd.DataFrame(vc_rows).sort_values('H2_BLUP',ascending=False)

blup.to_csv(os.path.join(OUTDIR,'BLUP_matrix.csv'))
vc_df.to_csv(os.path.join(OUTDIR,'VarianceComponents.csv'),index=False)

vc_df.head()

## Step 7 — Standardized BLUP Matrix

In [ ]:
Z = pd.DataFrame(
    StandardScaler().fit_transform(blup.values),
    columns=traits,
    index=blup.index
)

Z.to_csv(os.path.join(OUTDIR,'Standardized_BLUP.csv'))
Z.head()

## Step 8 — Partial Correlation (Graphical Lasso)

In [ ]:
gl = GraphicalLassoCV()
gl.fit(Z.values)

P = gl.precision_
d = np.sqrt(np.diag(P))

pcorr = -P / np.outer(d,d)
np.fill_diagonal(pcorr,1)

pcorr_df = pd.DataFrame(pcorr,index=traits,columns=traits)

pcorr_df.head()

## Step 9 — Partial Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,9))
plt.imshow(pcorr_df,cmap='coolwarm',vmin=-1,vmax=1)

labs=[safe_abbr(t) for t in traits]

plt.xticks(range(len(labs)),labs,rotation=45)
plt.yticks(range(len(labs)),labs)

plt.colorbar()

plt.title('BLUP Partial Correlation Heatmap')

plt.tight_layout()

plt.savefig(os.path.join(OUTDIR,'PartialCorrelation_heatmap.png'),dpi=600)

plt.show()

## Step 10 — Trait Interaction Network

In [ ]:
G = nx.Graph()

for i in range(len(traits)):
    for j in range(i+1,len(traits)):

        w = pcorr_df.iloc[i,j]

        if abs(w) >= THRESHOLD_PC:
            G.add_edge(traits[i],traits[j],weight=w)

pos = nx.spring_layout(G,seed=42)

plt.figure(figsize=(10,8))

nx.draw_networkx(G,pos,node_size=1500,font_size=9)

plt.title('Trait Interaction Network')

plt.savefig(os.path.join(OUTDIR,'TraitNetwork.png'),dpi=600)

plt.show()

## Step 11 — Yield Predictor Analysis

In [ ]:
yield_trait='Grain yield per hill'

X=Z.drop(columns=[yield_trait])
y=Z[yield_trait]

lin=LinearRegression().fit(X,y)

importance=pd.Series(abs(lin.coef_),index=X.columns).sort_values()

importance.plot.barh(figsize=(8,6))

plt.title('Yield Predictors')

plt.savefig(os.path.join(OUTDIR,'YieldPredictors.png'),dpi=600)

plt.show()

## Step 12 — Download Results

In [ ]:
zipfile = shutil.make_archive(OUTDIR,'zip',OUTDIR)
files.download(zipfile)